In [1]:
%load_ext autoreload
%autoreload 2

# Part 2: Content-Base Filtering with Ridge Regression
In this notebook, the mission is implementing a Content-Base Filtering Recommender System by treating each user as a separate machine learning problem to learn their preferences for different genres

**Methods to learn:**
* **Items Profile: ($X$):** Binary genres vectors
* **Users Profile: ($W$):** Learned weights represents genre preferences
* **Target ($Y$):** The ratings given by users
* **Regularization:** Using **Ridge Model** to prevent overfitting especially users who gave few ratings

## Import libraries

In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge 
from sklearn.metrics import mean_squared_error
from sklearn.feature_extraction.text import TfidfTransformer

## 1. Import files

### 1.1.Train-Test dataset

In [3]:
ratings_train=pd.read_csv('../data/processed/ratings_a_train.csv').values
ratings_test=pd.read_csv('../data/processed/ratings_a_test.csv').values

In [4]:
ratings_train

array([[        0,         0,         5, 874965758],
       [        0,         1,         3, 876893171],
       [        0,         2,         4, 878542960],
       ...,
       [      942,      1187,         3, 888640250],
       [      942,      1227,         3, 888640275],
       [      942,      1329,         3, 888692465]], shape=(90570, 4))

In [5]:
print(f'Number of traing rates: {ratings_train.shape[0]}')
print(f'Number of test rates: {ratings_test.shape[0]}')

Number of traing rates: 90570
Number of test rates: 9430


### 1.2.Items Profile

In [6]:
X_items_profile=np.load('../data/processed/items_profile.npy')

In [7]:
X_items_profile

array([[0, 0, 1, ..., 0, 0, 0],
       [1, 1, 0, ..., 1, 0, 0],
       [0, 0, 0, ..., 1, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(1682, 18))

## 2. Build Content-Based Recommender System

### 2.1.Build features vector

In [8]:
transfomer=TfidfTransformer(smooth_idf=True,norm='l2')
tfidf=transfomer.fit_transform(X_items_profile.tolist()).toarray()

In [9]:
tfidf

array([[0.        , 0.        , 0.74066017, ..., 0.        , 0.        ,
        0.        ],
       [0.53676706, 0.65097024, 0.        , ..., 0.53676706, 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 1.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ]], shape=(1682, 18))

### 2.2.Function to get movies that a user rated and the ratings of those movies 

In [10]:
def get_items_rated_by_user(rate_matrix,user_id):
    y_user=rate_matrix[:,0]
    ids=np.where(y_user==user_id)[0]
    items_id=rate_matrix[ids,1]
    scores=rate_matrix[ids,2]
    return (items_id,scores)

In [11]:
print(get_items_rated_by_user(ratings_train,0))

(array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  20,  21,  22,  23,  24,  25,  26,
        27,  28,  29,  30,  31,  33,  34,  35,  36,  37,  38,  39,  40,
        41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,  52,  53,
        54,  55,  56,  57,  58,  59,  61,  62,  63,  64,  65,  66,  67,
        68,  69,  70,  71,  72,  73,  74,  75,  76,  77,  78,  79,  80,
        81,  82,  83,  84,  85,  86,  87,  88,  89,  90,  91,  92,  93,
        94,  95,  96,  97,  98,  99, 100, 101, 102, 103, 104, 105, 106,
       107, 108, 109, 110, 111, 112, 113, 114, 115, 117, 118, 119, 120,
       121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133,
       134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146,
       147, 148, 149, 150, 151, 152, 153, 155, 156, 157, 158, 160, 161,
       162, 163, 164, 165, 166, 167, 168, 169, 171, 172, 173, 174, 175,
       176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 1

### 2.3.Build Content-Base Filtering System

**Using Ridge Regression to find weights and bias for each user**

Minimize this function:
$$L(\mathbf{w}_u)= \sum_{i \in R_u} (\mathbf{w}_u \mathbf{x}_i+b_u-y_{i,u})^2 + \lambda||\mathbf{w}_u||^2$$

**Using Linear Function to predict** 

Score-predicted $\hat{y}_{i,u}$ for movie $i$ and user $u$ 
$$\hat{y}_{i,u} = \mathbf{x}_i \cdot \mathbf{w}_u + b_u$$

**Meaning of parameters**
* $\mathbf{x}_i$: tfidf feature vector for movie i
* $\mathbf{w}_u$: weight vector for user u
* $b_u$: bias for user u
* $y_{i,u}$: real score of movie i rating by user u
* $\lambda$: parameter to prevent overfitting